In [1]:
from dotenv import load_dotenv
load_dotenv()

from ragwire import RAGWire, setup_logging
import ragwire
logger = setup_logging(log_level="INFO")

print(ragwire.__version__)

# rag = RAGWire("config_gemini.yaml")
# rag = RAGWire("config_openai.yaml")
rag = RAGWire("config_groq.yaml")

c:\Users\laxmi\anaconda3\envs\ml\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.1.0)/charset_normalizer (3.4.6) doesn't match a supported version!
  warnings.warn(


1.2.7
2026-03-26 18:50:36,666 - ragwire.core.pipeline - INFO - Loading configuration from config_groq.yaml
2026-03-26 18:50:37,045 - ragwire.core.pipeline - INFO - Document loader initialized
2026-03-26 18:50:37,046 - ragwire.core.pipeline - INFO - Text splitter initialized (strategy=markdown, chunk_size=10000)
2026-03-26 18:50:40,351 - ragwire.core.pipeline - INFO - Embedding model initialized (provider=huggingface)
2026-03-26 18:50:40,944 - ragwire.core.pipeline - INFO - LLM initialized for metadata extraction (provider=groq, model=qwen/qwen3-32b)
2026-03-26 18:50:41,468 - ragwire.vectorstores.qdrant_store - INFO - Connected to Qdrant at http://192.168.1.9:6333
2026-03-26 18:50:41,474 - ragwire.core.pipeline - INFO - Using existing collection: rag_documents_groq
2026-03-26 18:50:41,616 - ragwire.core.pipeline - INFO - Vector store initialized
2026-03-26 18:50:41,618 - ragwire.core.pipeline - INFO - Retriever initialized (type=hybrid, top_k=5, auto_filter=False)
2026-03-26 18:50:41,61

In [2]:
stats = rag.ingest_directory('../data')

2026-03-26 18:50:41,627 - ragwire.core.pipeline - INFO - Found 3 file(s) in ../data
2026-03-26 18:50:41,627 - ragwire.core.pipeline - INFO - Starting ingestion of 3 documents


Ingesting:   0%|          | 0/3 [00:00<?, ?file/s]

2026-03-26 18:50:41,638 - ragwire.core.pipeline - INFO - Skipping (already ingested): ..\data\amazon 10k 2025.pdf
2026-03-26 18:50:41,645 - ragwire.core.pipeline - INFO - Skipping (already ingested): ..\data\Apple_10k_2025.pdf
2026-03-26 18:50:41,651 - ragwire.core.pipeline - INFO - Skipping (already ingested): ..\data\GOOG-10-K-2025.pdf


Ingesting: 100%|██████████| 3/3 [00:00<00:00, 136.29file/s]

2026-03-26 18:50:41,788 - ragwire.core.pipeline - INFO - Ingestion complete: 0/3 documents


In [3]:
from typing import Optional

from langchain.agents import create_agent
from langchain.tools import tool
from langchain_core.messages import HumanMessage
from langchain_ollama import ChatOllama
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_openai import ChatOpenAI
from langchain_groq import ChatGroq
from langgraph.checkpoint.memory import InMemorySaver

# ------------------------------------------------------------------ #
# 2. Tools
# ------------------------------------------------------------------ #
@tool
def get_filter_context(query: str) -> str:
    """Get available metadata fields, stored values, and filter suggestions for a query.

    Call this before search_documents when the query involves specific metadata
    (company, year, document type, etc.). Use the returned context to decide
    what filters to pass to search_documents.

    Skip this for purely semantic queries with no metadata intent.
    """
    return rag.get_filter_context(query)


@tool
def search_documents(query: str, filters: Optional[dict] = None) -> str:
    """Search the document knowledge base for relevant information.

    Args:
        query: The search query
        filters: Optional metadata filters decided from get_filter_context.
                 Pass {} or omit to search without filtering.
    """
    results = rag.retrieve(query, top_k=1, filters=filters)
    if not results:
        return "No relevant documents found."

    chunks = []
    for doc in results:
        source = doc.metadata.get("file_name", "unknown")
        meta_parts = [
            f"{k}={str(v)[:100]}"
            for k, v in doc.metadata.items()
            if k != "file_name" and v not in (None, "", [])
        ]
        header = f"[{source}" + (f" | {', '.join(meta_parts)}" if meta_parts else "") + "]"
        chunks.append(f"{header}\n{doc.page_content}")

    return "\n\n---\n\n".join(chunks)


# ------------------------------------------------------------------ #
# 3. Agent with memory
# ------------------------------------------------------------------ #
# model = ChatOllama(model="qwen3.5:27b", base_url="http://localhost:11434")
# model = ChatGoogleGenerativeAI(model='gemini-3.1-flash-lite-preview')
# model = ChatOpenAI(model='gpt-5.4-nano')
model = ChatGroq(model='qwen/qwen3-32b')
checkpointer = InMemorySaver()

agent = create_agent(
    model=model,
    tools=[get_filter_context, search_documents],
    system_prompt=(
        "You are a helpful financial document assistant. "
        "For complex questions, break them down into simple sub-questions and answer each one before forming a final answer. "
        "Always use search_documents to retrieve information before answering — never answer from general knowledge. "
        "Use get_filter_context before search_documents when the query involves specific metadata (company, year, document type, etc.). "
        "If no relevant documents are found, say so — do not guess or fabricate an answer. "
        "Always cite the source document in your answer."
    ),
    checkpointer=checkpointer,
)



config = {"configurable": {"thread_id": "demo-1"}}


# ------------------------------------------------------------------ #
# 4. Interactive Q&A loop
# ------------------------------------------------------------------ #
print("\nRAG Agent ready. Type 'quit' to exit.\n")

while True:
    question = input("You: ").strip()
    if question.lower() in ("quit", "exit", "q"):
        break
    if not question:
        continue

    response = agent.invoke(
        {"messages": [HumanMessage(question)]},
        config=config,
    )
    print(f"\nAgent: {response['messages'][-1].text}\n\n\n")



RAG Agent ready. Type 'quit' to exit.

2026-03-26 18:50:51,945 - ragwire.core.pipeline - INFO - Auto-extracted filters from query: {'company_name': 'apple inc.'}
2026-03-26 18:50:52,753 - ragwire.core.pipeline - INFO - Retrieved 1 documents for query: Apple revenue...

Agent: Apple Inc.'s total net sales (revenue) for the 2025 fiscal year were **$416.16 billion** (as reported in the 2025 Form 10-K).



2026-03-26 18:51:48,127 - ragwire.core.pipeline - INFO - Auto-extracted filters from query: {'company_name': 'amazon.com inc.'}
2026-03-26 18:52:29,415 - ragwire.core.pipeline - INFO - Retrieved 1 documents for query: Amazon revenue...


APIStatusError: Error code: 413 - {'error': {'message': 'Request too large for model `qwen/qwen3-32b` in organization `org_01kmn3r6qhe7m8trd5a55y0mj9` service tier `on_demand` on tokens per minute (TPM): Limit 6000, Requested 6993, please reduce your message size and try again. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}